<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>

----

### Вариант задания 9


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

[ваш текст]

#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) и реализуйте полиморфизм с перекрытием и прегегрузкой методов, а также generic классы

<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [2]:
using System;
using System.Collections.Generic;

// Товар
public class Item
{
    public string Name { get; set; }
    public decimal Price { get; set; }
    
    public Item(string name, decimal price)
    {
        Name = name;
        Price = price;
    }
}

// === БАЗОВЫЙ КЛАСС ===
public class Order
{
    private decimal _totalAmount;

    public int OrderId { get; set; }
    public DateTime CreationDate { get; set; }
    public List<Item> Items { get; set; } = new List<Item>();

    // Свойство с геттером и сеттером (с проверкой)
    public decimal TotalAmount
    {
        get => _totalAmount;
        set
        {
            if (value < 0) throw new Exception("Сумма не может быть отрицательной");
            _totalAmount = value;
        }
    }

    public Order(int orderId)
    {
        OrderId = orderId;
        CreationDate = DateTime.Now;
    }

    // Перекрываемый метод
    public virtual decimal CalculateTotal()
    {
        TotalAmount = 0;
        foreach (var item in Items) TotalAmount += item.Price;
        return TotalAmount;
    }

    // Перегрузка метода CalculateTotal (со скидкой)
    public virtual decimal CalculateTotal(decimal discount)
    {
        TotalAmount = CalculateTotal() - discount;
        return TotalAmount;
    }

    public virtual void AddItem(Item item)
    {
        Items.Add(item);
        Console.WriteLine($"Заказ #{OrderId}: добавлен {item.Name}");
    }

    // Перегрузка AddItem (добавить сразу несколько)
    public virtual void AddItem(Item item, int count)
    {
        for (int i = 0; i < count; i++) Items.Add(item);
        Console.WriteLine($"Заказ #{OrderId}: добавлено {count} шт. {item.Name}");
    }

    public virtual void RemoveItem(Item item)
    {
        Items.Remove(item);
        Console.WriteLine($"Заказ #{OrderId}: удален {item.Name}");
    }

    public virtual void ShowInfo()
    {
        Console.WriteLine($"Заказ #{OrderId} | Сумма: {TotalAmount}");
    }
}

// === ОНЛАЙН ЗАКАЗ ===
public class OnlineOrder : Order
{
    public string CustomerEmail { get; set; }

    public OnlineOrder(int orderId, string email) : base(orderId)
    {
        CustomerEmail = email;
    }

    // Перекрытие метода
    public override void AddItem(Item item)
    {
        base.AddItem(item);
        Console.WriteLine($"  -> Отправим чек на {CustomerEmail}");
    }

    public override void ShowInfo()
    {
        Console.WriteLine($"Онлайн-заказ #{OrderId} | Email: {CustomerEmail} | Сумма: {TotalAmount}");
    }
}

// === ФИЗИЧЕСКИЙ ЗАКАЗ ===
public class PhysicalOrder : Order
{
    public string DeliveryAddress { get; set; }

    public PhysicalOrder(int orderId, string address) : base(orderId)
    {
        DeliveryAddress = address;
    }

    // Перекрытие метода
    public override void RemoveItem(Item item)
    {
        base.RemoveItem(item);
        Console.WriteLine($"  -> Оформлен возврат по адресу: {DeliveryAddress}");
    }

    public override void ShowInfo()
    {
        Console.WriteLine($"Физ. заказ #{OrderId} | Адрес: {DeliveryAddress} | Сумма: {TotalAmount}");
    }
}

// === СПЕЦИАЛИЗИРОВАННЫЙ ЗАКАЗ ===
public class SpecializedOrder : Order
{
    public string SpecialConditions { get; set; }

    public SpecializedOrder(int orderId, string conditions) : base(orderId)
    {
        SpecialConditions = conditions;
    }

    // Перекрытие метода (добавляем спец. скидку)
    public override decimal CalculateTotal()
    {
        base.CalculateTotal();
        TotalAmount = TotalAmount * 0.9m; // скидка 10%
        Console.WriteLine($"  -> Применены спец. условия: {SpecialConditions} (скидка 10%)");
        return TotalAmount;
    }

    public override void ShowInfo()
    {
        Console.WriteLine($"Спец. заказ #{OrderId} | Условия: {SpecialConditions} | Сумма: {TotalAmount}");
    }
}

// === GENERIC КЛАСС ===
public class OrderCollection<T> where T : Order
{
    private List<T> list = new List<T>();

    public void Add(T order) => list.Add(order);

    public void ShowAll()
    {
        foreach (var o in list)
        {
            o.ShowInfo();
        }
    }
}

// ==========================================
// ПРОВЕРКА РАБОТЫ
// ==========================================
Console.WriteLine("=== 1. Онлайн-заказ ===");
var online = new OnlineOrder(1, "test@mail.ru");
online.AddItem(new Item("Книга", 500));
online.AddItem(new Item("Ручка", 50), 3); // перегрузка
online.CalculateTotal();
online.ShowInfo();

Console.WriteLine("\n=== 2. Физический заказ ===");
var physical = new PhysicalOrder(2, "ул. Пушкина, 10");
var phone = new Item("Телефон", 15000);
physical.AddItem(phone);
physical.CalculateTotal();
physical.RemoveItem(phone); // перекрытие
physical.ShowInfo();

Console.WriteLine("\n=== 3. Специализированный заказ ===");
var special = new SpecializedOrder(3, "Опт");
special.AddItem(new Item("Стол", 10000));
special.CalculateTotal(); // перекрытие
special.ShowInfo();

Console.WriteLine("\n=== 4. Generic-класс ===");
var collection = new OrderCollection<Order>();
collection.Add(online);
collection.Add(physical);
collection.Add(special);
collection.ShowAll();

=== 1. Онлайн-заказ ===
Заказ #1: добавлен Книга
  -> Отправим чек на test@mail.ru
Заказ #1: добавлено 3 шт. Ручка
Онлайн-заказ #1 | Email: test@mail.ru | Сумма: 650

=== 2. Физический заказ ===
Заказ #2: добавлен Телефон
Заказ #2: удален Телефон
  -> Оформлен возврат по адресу: ул. Пушкина, 10
Физ. заказ #2 | Адрес: ул. Пушкина, 10 | Сумма: 15000

=== 3. Специализированный заказ ===
Заказ #3: добавлен Стол
  -> Применены спец. условия: Опт (скидка 10%)
Спец. заказ #3 | Условия: Опт | Сумма: 9000.0

=== 4. Generic-класс ===
Онлайн-заказ #1 | Email: test@mail.ru | Сумма: 650
Физ. заказ #2 | Адрес: ул. Пушкина, 10 | Сумма: 15000
Спец. заказ #3 | Условия: Опт | Сумма: 9000.0
